# Douban Dataset Preprocessing for Cross-Domain Recommendation (CDR)

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn import preprocessing as pp
from sklearn.model_selection import train_test_split

# Display configurations
pd.set_option('display.max_colwidth', None)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

def save_mapping(sfile, domain_name, item_col):
    # Read raw data using tab/whitespace separation
    df = pd.read_csv(sfile, sep=r'\s+')
    df = df[['user_id', item_col, 'rating']]
    df.columns = ['user_id', 'item_id', 'rating']
    
    print(f"--- Processing Domain: {domain_name} ---")
    print(f"Before filtering: Total interactions = {len(df)}")
    
    # 5-core filtering: Keep users and items with >= 5 interactions
    user_counts = df['user_id'].value_counts()
    df = df[df['user_id'].isin(user_counts[user_counts >= 5].index)]
    
    item_counts = df['item_id'].value_counts()
    df = df[df['item_id'].isin(item_counts[item_counts >= 5].index)]
    
    print(f"After filtering: Total interactions = {len(df)}")
    
    # User and Item Label Encoding
    le_user = pp.LabelEncoder()
    le_item = pp.LabelEncoder()
    df['user_id_idx'] = le_user.fit_transform(df['user_id'].values)
    df['item_id_idx'] = le_item.fit_transform(df['item_id'].values)
    
    n_users = df['user_id_idx'].nunique()
    n_items = df['item_id_idx'].nunique()
    print(f"Unique Users: {n_users} | Unique Items: {n_items}")
    
    # Save User Mapping
    os.makedirs(os.path.dirname(user_map_paths[domain_name]), exist_ok=True)
    with open(user_map_paths[domain_name], "w") as output:
        for k, v in zip(le_user.classes_, le_user.transform(le_user.classes_)):
            output.write(f'{k} {v}\n')
            
    # Save Item Mapping
    os.makedirs(os.path.dirname(item_map_paths[domain_name]), exist_ok=True)
    with open(item_map_paths[domain_name], "w") as output:
        for k, v in zip(le_item.classes_, le_item.transform(le_item.classes_)):
            output.write(f'{k} {v}\n')
            
    print(f"✅ Saved mappings for {domain_name}\n")
    return n_users, n_items, df

# Execute mapping for both domains
movie_n_users, movie_n_items, movie_dataframe = save_mapping(douban_movie, "douban_movie", "movie_id")
music_n_users, music_n_items, music_dataframe = save_mapping(douban_music, "douban_music", "music_id")

In [ ]:
def save_overlap(domain_a, domain_b, df_a, df_b, filename):
    # Extract unique users and their mapped indices
    user_a = df_a[['user_id', 'user_id_idx']].drop_duplicates()
    user_b = df_b[['user_id', 'user_id_idx']].drop_duplicates()
    
    # Inner join on original user_id to find overlap
    overlap = pd.merge(user_a, user_b, on="user_id", how="inner", suffixes=(f"_{domain_a}", f"_{domain_b}"))
    out_df = overlap[[f"user_id_idx_{domain_a}", f"user_id_idx_{domain_b}"]]
    
    # Save overlap text file
    os.makedirs(douban_overlap_save, exist_ok=True)
    out_path = os.path.join(douban_overlap_save, filename)
    out_df.to_csv(out_path, sep=' ', index=False, header=False)
    
    print(f"✅ Saved {len(out_df)} overlaps -> {out_path}")
    return out_df

# Generate overlap records for both directions
movie_music_overlap = save_overlap("movie", "music", movie_dataframe, music_dataframe, "movie-music-overlap.txt")
music_movie_overlap = save_overlap("music", "movie", music_dataframe, movie_dataframe, "music-movie-overlap.txt")

In [ ]:
def split_and_save(ratio, overlap_filename, target_df, total_n_users, total_n_items, total_tn_users, total_tn_items, save_dir):
    overlap_path = os.path.join(douban_overlap_save, overlap_filename)
    print(f"Loading overlap users from: {overlap_path}")
    
    # Load mapped overlap user coordinates (mapped_id_s, mapped_id_t)
    overlap_df = pd.read_csv(overlap_path, sep=' ', header=None, names=["mapped_id_s", "mapped_id_t"])
    overlap_user_indices = overlap_df['mapped_id_t'].unique()
    print(f"Total number of overlap users: {len(overlap_user_indices)}")
    
    # Filter target interactions belonging to overlap users
    overlap_df_interactions = target_df[target_df['user_id_idx'].isin(overlap_user_indices)].copy()
    print(f"Total interactions for overlap users: {len(overlap_df_interactions)}")
    
    all_overlap_users = overlap_df_interactions['user_id_idx'].unique()
    
    # Split overlap users into train/test sets
    train_users, test_users = train_test_split(
        all_overlap_users, 
        test_size=(1 - ratio), 
        random_state=42
    )
    
    print(f"\n--- Splitting with ratio {ratio*100:.0f}% ---")
    print(f"Train users: {len(train_users)} | Test users: {len(test_users)}")
    
    train_df = overlap_df_interactions[overlap_df_interactions['user_id_idx'].isin(train_users)]
    test_df = overlap_df_interactions[overlap_df_interactions['user_id_idx'].isin(test_users)]
    
    print(f"Train interactions: {len(train_df)} | Test interactions: {len(test_df)}")
    
    os.makedirs(save_dir, exist_ok=True)
    
    # 1. Save training dataframe as a space-separated .txt file
    txt_filename = f"{int(ratio*100)}train.txt"
    txt_path = os.path.join(save_dir, txt_filename)
    train_df[['user_id_idx', 'item_id_idx', 'rating']].to_csv(txt_path, sep=' ', index=False, header=False)
    print(f"✅ Saved training interactions to: {txt_path}")
    
    # 2. Convert datasets to PyTorch arrays and dictionary structure
    train_uid = train_df['user_id_idx'].values.astype(np.int64)
    train_iid = train_df['item_id_idx'].values.astype(np.int64)
    train_rates = train_df['rating'].values.astype(np.float32)

    test_uid = test_df['user_id_idx'].values.astype(np.int64)
    test_iid = test_df['item_id_idx'].values.astype(np.int64)
    test_rates = test_df['rating'].values.astype(np.float32)

    data_to_save = {
        'train_uid': torch.from_numpy(train_uid),
        'train_iid': torch.from_numpy(train_iid),
        'train_rates': torch.from_numpy(train_rates),
        'test_uid': torch.from_numpy(test_uid),
        'test_iid': torch.from_numpy(test_iid),
        'test_rates': torch.from_numpy(test_rates),
        'n_user': total_n_users, 
        'n_item': total_n_items,
        'tn_user': total_tn_users, 
        'tn_item': total_tn_items
    }

    # Save PyTorch data object
    pt_filename = f"{int(ratio*100)}train.pt"
    pt_path = os.path.join(save_dir, pt_filename)
    torch.save(data_to_save, pt_path)
    print(f"✅ Saved PyTorch data to: {pt_path}\n")

# Process splits for 20% and 50% Movie -> Music Cross Domain Recommendation
split_and_save(
    ratio=0.2, 
    overlap_filename="movie-music-overlap.txt",
    target_df=music_dataframe, 
    total_n_users=movie_n_users, 
    total_n_items=movie_n_items, 
    total_tn_users=music_n_users, 
    total_tn_items=music_n_items,
    save_dir=train_data["douban_movie_music_save"]
)

split_and_save(
    ratio=0.5, 
    overlap_filename="movie-music-overlap.txt",
    target_df=music_dataframe, 
    total_n_users=movie_n_users, 
    total_n_items=movie_n_items, 
    total_tn_users=music_n_users, 
    total_tn_items=music_n_items,
    save_dir=train_data["douban_movie_music_save"]
)